# RSICD Adapter-CLIP — Colab Quickstart

End-to-end reproduction in 7 cells. Designed for Google Colab T4 GPU.

**What this notebook does:**
1. Mounts Drive (for backup)
2. Installs dependencies
3. Clones the repo and downloads the RSICD dataset
4. Smoke test: verify GPU + dataset
5. Run zero-shot baseline (5 min)
6. Train the adapter (~2 hours)
7. Inspect results and back up to Drive

## Cell 1 — Setup: mount Drive + install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install open_clip_torch faiss-cpu ftfy accelerate pyyaml -q

import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
print("Environment ready.")

## Cell 2 — Clone repo and download data

In [ ]:
# Clone the repo
!git clone https://github.com/YOUR_USERNAME/rsicd-clip-adapter.git
%cd rsicd-clip-adapter

# Option A: Kaggle (requires kaggle.json uploaded to Files panel)
# !pip install kaggle -q
# !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d thedevastator/rsicd-image-caption-dataset --path data/raw --unzip

# Option B: HuggingFace redistribution (if you uploaded a tar of archive/)
# !tar -xf /content/drive/MyDrive/archive.tar
# !python scripts/00b_prepare_rsicd.py

# Option C: manual upload (zip the data/raw/ folder and upload through the Files panel)
# then just unzip:
# !unzip -q /content/data_raw.zip -d data/

!ls data/raw/ | head

## Cell 3 — Smoke test: verify GPU + dataset

In [ ]:
import sys; sys.path.insert(0, '.')
import torch
import open_clip
from src.utils import get_device

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name:      {torch.cuda.get_device_name(0)}")
print(f"Chosen device: {get_device()}")

model, preprocess = open_clip.create_model_from_pretrained(
    'ViT-B-32', pretrained='openai', force_quick_gelu=True
)
tokenizer = open_clip.get_tokenizer('ViT-B-32')
print(f"CLIP loaded. Params: {sum(p.numel() for p in model.parameters()):,}")

from src.dataset import get_dataloaders
train_loader, val_ret, test_ret, _, _ = get_dataloaders(
    "data/splits", "data/raw/RSICD_images", batch_size=4
)
images, captions, ids = next(iter(train_loader))
print(f"Image shape:   {images.shape}")
print(f"Caption shape: {captions.shape}")
print("Smoke test PASSED")

## Cell 4 — Run zero-shot baseline (5 min)

In [ ]:
!python scripts/01_prepare_splits.py
!python scripts/02_run_baseline.py

## Cell 5 — Train the adapter (~2 hours on T4)

In [ ]:
# ~2 hours on Colab T4 for the full 20-epoch run
!python scripts/03_train_adapter.py configs/adapter_base.yaml adapter

# Optional: in parallel, start the full fine-tune in another notebook
# !python scripts/04_run_fullfinetune.py

## Cell 6 — Inspect results

In [ ]:
import json
with open("results/metrics/baseline_zeroshot.json") as f:
    zs = json.load(f)
with open("results/metrics/adapter_results.json") as f:
    ad = json.load(f)

print("Zero-shot CLIP:")
print(f"  T->I  R@1={zs['text_to_image']['R@1']:>5.2f}  R@5={zs['text_to_image']['R@5']:>5.2f}  R@10={zs['text_to_image']['R@10']:>5.2f}")
print()
print("Adapter-CLIP (ours):")
print(f"  T->I  R@1={ad['text_to_image']['R@1']:>5.2f}  R@5={ad['text_to_image']['R@5']:>5.2f}  R@10={ad['text_to_image']['R@10']:>5.2f}")
print()
print(f"Improvement: +{ad['text_to_image']['R@1'] - zs['text_to_image']['R@1']:.1f} pp on T->I R@1")
print(f"Trainable:   {ad['trainable_params']:,} ({100*ad['trainable_params']/ad['total_params']:.2f}% of CLIP)")

## Cell 7 — Save results to Google Drive (don't lose your work!)

In [ ]:
import shutil
DRIVE_DIR = "/content/drive/MyDrive/rsicd_results"
shutil.copytree("results", DRIVE_DIR, dirs_exist_ok=True)
print(f"Backed up to {DRIVE_DIR}")

# Also copy the paper figures
FIG_DIR = "/content/drive/MyDrive/rsicd_figures"
shutil.copytree("paper/figures", FIG_DIR, dirs_exist_ok=True)
print(f"Figures backed up to {FIG_DIR}")